# NCAA March Madness Stacking & Automated Tuning (v4)

This notebook focuses on:
1. **Multiple Base Models**: XGBoost and LightGBM.
2. **Hyperparameter Optimization**: Setting refined parameters.
3. **Model Stacking**: Using a Meta-Learner (Logistic Regression) to combine model predictions.
4. **Win Percentage Feature**: Adding seasonal win rate for both teams.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler
import os

DATA_PATH = './march-machine-learning-mania-2026/'
team_features = pd.read_csv('all_team_features_v2.csv')

m_tourney_results = pd.read_csv(os.path.join(DATA_PATH, 'MNCAATourneyDetailedResults.csv'))
w_tourney_results = pd.read_csv(os.path.join(DATA_PATH, 'WNCAATourneyDetailedResults.csv'))
tourney_results = pd.concat([m_tourney_results, w_tourney_results], ignore_index=True)

print("Data loaded.")

## 1. Feature Enhancement: Win Percentage

Adding overall winning percentage to the feature set.

In [ ]:
def add_win_percentage(gender, features_df):
    prefix = gender.upper()
    reg_results = pd.read_csv(os.path.join(DATA_PATH, f'{prefix}RegularSeasonDetailedResults.csv'))
    
    # Win/Loss counts
    wins = reg_results.groupby(['Season', 'WTeamID']).size().reset_index(name='Wins')
    losses = reg_results.groupby(['Season', 'LTeamID']).size().reset_index(name='Losses')
    
    wins = wins.rename(columns={'WTeamID': 'TeamID'})
    losses = losses.rename(columns={'LTeamID': 'TeamID'})
    
    stats = pd.merge(wins, losses, on=['Season', 'TeamID'], how='outer').fillna(0)
    stats['WinPct'] = stats['Wins'] / (stats['Wins'] + stats['Losses'])
    
    return features_df.merge(stats[['Season', 'TeamID', 'WinPct']], on=['Season', 'TeamID'], how='left').fillna(0.5)

m_feat = team_features[team_features['TeamID'] < 3000].copy()
w_feat = team_features[team_features['TeamID'] >= 3000].copy()

m_feat = add_win_percentage('M', m_feat)
w_feat = add_win_percentage('W', w_feat)
team_features = pd.concat([m_feat, w_feat])
print("Win percentage feature added.")

## 2. Augmented Training Data Preparation

In [ ]:
def prepare_training_data_v4(results_df, features_df):
    training_rows = []
    feat_dict = features_df.set_index(['Season', 'TeamID']).to_dict('index')
    cols = ['SeedInt', 'OffEff', 'DefEff', 'SOS_Off', 'SOS_Def', 'TrendOffEff', 'TrendScore', 'AvgRank', 'WinPct']
    
    for i, row in results_df.iterrows():
        season, w_team, l_team = row['Season'], row['WTeamID'], row['LTeamID']
        try:
            wf, lf = feat_dict[(season, w_team)], feat_dict[(season, l_team)]
            # Symmetric rows
            training_rows.append({**{f'{c}Diff': wf[c] - lf[c] for c in cols}, 'label': 1, 'Season': season})
            training_rows.append({**{f'{c}Diff': lf[c] - wf[c] for c in cols}, 'label': 0, 'Season': season})
        except KeyError: continue
    return pd.DataFrame(training_rows)

train_df = prepare_training_data_v4(tourney_results, team_features)
print(f"Training set ready: {len(train_df)} rows.")

## 3. Stacking: XGBoost + LightGBM

We use Out-of-Fold (OOF) predictions to combine the models without overfitting.

In [ ]:
X = train_df.drop(['Season', 'label'], axis=1)
y = train_df['label']

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_xgb, oof_lgb = np.zeros(len(X)), np.zeros(len(X))

params_xgb = {'eval_metric': 'logloss', 'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 150, 'subsample': 0.8, 'random_state': 42}
params_lgb = {'objective': 'binary', 'metric': 'binary_logloss', 'num_leaves': 16, 'learning_rate': 0.05, 'n_estimators': 150, 'random_state': 42}

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # XGBoost
    m_xgb = xgb.XGBClassifier(**params_xgb)
    m_xgb.fit(X_train, y_train)
    oof_xgb[val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    
    # LightGBM
    m_lgb = lgb.LGBMClassifier(**params_lgb, verbose=-1)
    m_lgb.fit(X_train, y_train)
    oof_lgb[val_idx] = m_lgb.predict_proba(X_val)[:, 1]

print(f"XGB OOF Log Loss: {log_loss(y, oof_xgb):.4f}")
print(f"LGB OOF Log Loss: {log_loss(y, oof_lgb):.4f}")

# Simple Weighted Average for final stacking
ensemble_oof = 0.5 * oof_xgb + 0.5 * oof_lgb
print(f"Staked Ensemble OOF Log Loss: {log_loss(y, ensemble_oof):.4f}")

## 4. Final Submission Generation

In [ ]:
# Pre-train on full data
final_xgb = xgb.XGBClassifier(**params_xgb)
final_xgb.fit(X, y)
final_lgb = lgb.LGBMClassifier(**params_lgb, verbose=-1)
final_lgb.fit(X, y)

submission = pd.read_csv(os.path.join(DATA_PATH, 'SampleSubmissionStage1.csv'))
feat_lookup = team_features.set_index(['Season', 'TeamID']).to_dict('index')
cols = ['SeedInt', 'OffEff', 'DefEff', 'SOS_Off', 'SOS_Def', 'TrendOffEff', 'TrendScore', 'AvgRank', 'WinPct']

def get_submission_pred_v4(row):
    parts = row['ID'].split('_')
    season, t1, t2 = int(parts[0]), int(parts[1]), int(parts[2])
    try:
        f1, f2 = feat_lookup[(season, t1)], feat_lookup[(season, t2)]
        feat_row = pd.DataFrame([{f'{c}Diff': f1[c] - f2[c] for c in cols}])
        p_xgb = final_xgb.predict_proba(feat_row)[:, 1][0]
        p_lgb = final_lgb.predict_proba(feat_row)[:, 1][0]
        return np.clip(0.5 * p_xgb + 0.5 * p_lgb, 0.02, 0.98)
    except KeyError: return 0.5

submission['Pred'] = submission.apply(get_submission_pred_v4, axis=1)
submission.to_csv('submission_v4.csv', index=False)
print("Submission v4 complete!")